<a href="https://colab.research.google.com/github/shahaanabbas-13/Quantum-Diabetes-Prediction/blob/main/notebooks/05_Quantum_Kernel_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 — Quantum Kernel SVM (QSVM)
A small-sample proof-of-concept using PennyLane's CPU simulator. Quantum kernel evaluation is expensive (quadratic in sample count), so start with a stratified training subset. Keep the same held-out test split for final comparisons and clearly report sample counts.

In [1]:
%pip -q install pennylane scikit-learn

In [2]:
import pennylane as qml
print("PennyLane:",qml.__version__)

PennyLane: 0.45.1


In [3]:
from pathlib import Path
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif, SelectKBest, RFE
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    fbeta_score, roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve)
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd()
# If running from a notebook in /notebooks, move to project root.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "dataset" / "diabetes.csv"
if not DATA_PATH.exists():
    # Colab fallback: upload diabetes.csv into the current session or mount Drive.
    DATA_PATH = Path("/content/diabetes.csv")
df = pd.read_csv(DATA_PATH)
TARGET = "Outcome"
FEATURES = [c for c in df.columns if c != TARGET]
INVALID_ZERO_COLS = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.svm import SVC
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.utils import resample
X=df.drop(columns=TARGET).replace({c:{0:np.nan} for c in INVALID_ZERO_COLS})
y=df[TARGET].astype(int)
Xtr_raw,Xte_raw,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
# Fit preprocessing and MI selection on training only
imp=KNNImputer(n_neighbors=5,weights="distance")
Xtr=imp.fit_transform(Xtr_raw); Xte=imp.transform(Xte_raw)
sc=StandardScaler(); Xtr=sc.fit_transform(Xtr); Xte=sc.transform(Xte)
selector=SelectKBest(mutual_info_classif,k=4)
Xtr=selector.fit_transform(Xtr,y_train); Xte=selector.transform(Xte)
# Quantum angle embedding expects bounded inputs
mm=MinMaxScaler(feature_range=(-np.pi,np.pi))
Xtr=mm.fit_transform(Xtr); Xte=mm.transform(Xte)
# Stratified subset for kernel training
if len(Xtr) > 250:
    _, Xq, _, yq = train_test_split(
        Xtr, np.asarray(y_train), test_size=250, stratify=np.asarray(y_train),
        random_state=42
    )
else:
    Xq, yq = Xtr, np.asarray(y_train)
yq=np.asarray(yq)
n_qubits=4
dev=qml.device("default.qubit",wires=n_qubits)
def feature_map(x):
    qml.AngleEmbedding(x,wires=range(n_qubits),rotation="Y")
    for i in range(n_qubits-1):
        qml.CNOT(wires=[i,i+1])
    qml.CNOT(wires=[n_qubits-1,0])
@qml.qnode(dev)
def kernel_circuit(x1,x2):
    feature_map(x1)
    qml.adjoint(feature_map)(x2)
    return qml.probs(wires=range(n_qubits))
def quantum_kernel(A,B):
    K=np.zeros((len(A),len(B)))
    for i,a in enumerate(A):
        for j,b in enumerate(B):
            K[i,j]=kernel_circuit(a,b)[0]
    return K

print("Total dataset samples:", len(df))
print("Training samples:", len(Xtr_raw))
print("Test samples:", len(Xte_raw))
print("QSVM training samples:", len(Xq))
print("QSVM test samples:", len(Xte))
print("Number of qubits:", n_qubits)
print("Number of selected features:", Xq.shape[1])


# ==========================================
# QSVM TRAINING, EVALUATION AND RESULT SAVING
# ==========================================

experiment_start = time.time()

# 1. Generate quantum kernel matrices
print("Calculating training quantum kernel...")
K_train = quantum_kernel(Xq, Xq)

print("Calculating testing quantum kernel...")
K_test = quantum_kernel(Xte, Xq)

# 2. Train QSVM
print("Training QSVM...")
qsvm = SVC(
    kernel="precomputed",
    class_weight="balanced"
)

qsvm.fit(K_train, yq)

# 3. Generate predictions
pred = qsvm.predict(K_test)
scores = qsvm.decision_function(K_test)

# 4. Calculate execution time
execution_time = time.time() - experiment_start

# 5. Calculate evaluation metrics
accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred, zero_division=0)
recall = recall_score(y_test, pred, zero_division=0)
f1 = f1_score(y_test, pred, zero_division=0)
f2 = fbeta_score(y_test, pred, beta=2, zero_division=0)
roc_auc = roc_auc_score(y_test, scores)
pr_auc = average_precision_score(y_test, scores)

tn, fp, fn, tp = confusion_matrix(
    y_test, pred, labels=[0, 1]
).ravel()

# 6. Print classification report
print("\n========== QSVM CLASSIFICATION REPORT ==========")
print(classification_report(y_test, pred, digits=3))

print("ROC-AUC:", roc_auc)
print("PR-AUC:", pr_auc)
print("Execution time:", round(execution_time, 2), "seconds")

# 7. Prepare structured experiment results
selected_features = list(
    np.array(FEATURES)[selector.get_support()]
)

qsvm_results = pd.DataFrame([{
    "model": "QSVM",
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1_score": f1,
    "f2_score": f2,
    "roc_auc": roc_auc,
    "pr_auc": pr_auc,
    "specificity": tn / (tn + fp) if (tn + fp) else 0,
    "true_negative": tn,
    "false_positive": fp,
    "false_negative": fn,
    "true_positive": tp,
    "total_samples": len(df),
    "training_samples": len(Xtr_raw),
    "quantum_training_samples": len(Xq),
    "test_samples": len(Xte),
    "number_of_qubits": n_qubits,
    "number_of_features": Xq.shape[1],
    "selected_features": ", ".join(selected_features),
    "imputation": "KNN",
    "feature_selection": "Mutual Information",
    "kernel": "Fidelity Quantum Kernel",
    "execution_time_seconds": round(execution_time, 2),
    "random_state": RANDOM_STATE
}])

# 8. Display results
print("\n========== QSVM EXPERIMENT SUMMARY ==========")
display(qsvm_results)

# 9. Save results in Colab
RESULTS_DIR = Path("/content/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

QSVM_PATH = RESULTS_DIR / "qsvm_results.csv"

qsvm_results.to_csv(QSVM_PATH, index=False)

print("\nResults saved successfully!")
print("Location:", QSVM_PATH)

Total dataset samples: 768
Training samples: 614
Test samples: 154
QSVM training samples: 250
QSVM test samples: 154
Number of qubits: 4
Number of selected features: 4
Calculating training quantum kernel...
Calculating testing quantum kernel...
Training QSVM...

========== QSVM CLASSIFICATION REPORT ==========
              precision    recall  f1-score   support

           0      0.835     0.710     0.768       100
           1      0.580     0.741     0.650        54

    accuracy                          0.721       154
   macro avg      0.708     0.725     0.709       154
weighted avg      0.746     0.721     0.726       154

ROC-AUC: 0.7975925925925926
PR-AUC: 0.6601875239696877
Execution time: 498.38 seconds

========== QSVM EXPERIMENT SUMMARY ==========


,model,accuracy,precision,recall,f1_score,f2_score,roc_auc,pr_auc,specificity,true_negative,...,quantum_training_samples,test_samples,number_of_qubits,number_of_features,selected_features,imputation,feature_selection,kernel,execution_time_seconds,random_state
0,QSVM,0.720779,0.57971,0.740741,0.650407,0.701754,0.797593,0.660188,0.71,71,...,250,154,4,4,"Glucose, Insulin, BMI, Age",KNN,Mutual Information,Fidelity Quantum Kernel,498.38,42



Results saved successfully!
Location: /content/results/qsvm_results.csv


If kernel evaluation is too slow, reduce the training subset to 100–150 samples for a smoke test. Report this limitation honestly. The kernel is computed from state overlap probabilities; no quantum speedup is claimed.

In [4]:

# ==========================================
# UPLOAD QSVM RESULTS TO GITHUB
# ==========================================

import requests
import base64
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

OWNER = "shahaanabbas-13"
REPO = "Quantum-Diabetes-Prediction"
BRANCH = "main"

github_path = "results/qsvm_results.csv"

# Read local CSV
with open(QSVM_PATH, "rb") as f:
    content = base64.b64encode(f.read()).decode("utf-8")

api_url = (
    f"https://api.github.com/repos/{OWNER}/{REPO}"
    f"/contents/{github_path}"
)

headers = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json"
}

# Check whether file already exists
check = requests.get(
    api_url,
    headers=headers,
    params={"ref": BRANCH}
)

payload = {
    "message": "Add or update QSVM experiment results",
    "content": content,
    "branch": BRANCH
}

if check.status_code == 200:
    payload["sha"] = check.json()["sha"]
elif check.status_code != 404:
    raise Exception(f"GitHub check failed: {check.text}")

# Upload or update
response = requests.put(
    api_url,
    headers=headers,
    json=payload
)

response.raise_for_status()

print("QSVM results uploaded successfully to GitHub!")
print("Repository:", REPO)
print("File:", github_path)
print("Status:", response.status_code)

QSVM results uploaded successfully to GitHub!
Repository: Quantum-Diabetes-Prediction
File: results/qsvm_results.csv
Status: 201
